In [2]:
#Importing Libraries
import json
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_huggingface import  ChatHuggingFace, HuggingFaceEndpoint
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate


import os
from dotenv import load_dotenv


In [3]:
#Load environment variable api key
load_dotenv()

True

# Load and View the HR FAQ Dataset(Data Ingestion)

In [4]:
# Load Json HR faq dataset
with open("D:/QA-System/hr_faq_data.json", "r") as f:
    data = json.load(f)

print("Loaded sucessfully")    



Loaded sucessfully


In [5]:
from pprint import pprint

pprint(data)

[{'answer': 'Our company offers a hybrid work model. Employees can work '
            'remotely up to 3 days per week, with in-office presence required '
            'on Tuesdays and Thursdays for team meetings and collaboration. '
            'Department heads may adjust this policy based on project needs. '
            'Employees must ensure they have reliable internet connection and '
            'a secure workspace when working remotely.',
  'category': 'remote_work',
  'question': "What is the company's policy on remote work?"},
 {'answer': 'Full-time employees receive 15 paid vacation days annually, '
            'accrued at 1.25 days per month. After 3 years of service, this '
            'increases to 20 days. Unused vacation days (up to 5) can be '
            'carried over to the next calendar year. Vacation requests must be '
            'submitted at least 2 weeks in advance through the HR portal.',
  'category': 'leave_policy',
  'question': 'How many vacation days am I en

In [6]:
#preprocess QA data
def preprocess(text):
    return text.lower().strip()


In [7]:
documents = []
for entry in data:
    question = preprocess(entry["question"])
    answer = preprocess(entry["answer"])
    category = entry.get("category", "")
    full_text = f"Question: {question}\nAnswer: {answer}"
    doc = Document(page_content=full_text, metadata={"category": category})
    documents.append(doc)


In [8]:
documents

[Document(metadata={'category': 'remote_work'}, page_content="Question: what is the company's policy on remote work?\nAnswer: our company offers a hybrid work model. employees can work remotely up to 3 days per week, with in-office presence required on tuesdays and thursdays for team meetings and collaboration. department heads may adjust this policy based on project needs. employees must ensure they have reliable internet connection and a secure workspace when working remotely."),
 Document(metadata={'category': 'leave_policy'}, page_content='Question: how many vacation days am i entitled to per year?\nAnswer: full-time employees receive 15 paid vacation days annually, accrued at 1.25 days per month. after 3 years of service, this increases to 20 days. unused vacation days (up to 5) can be carried over to the next calendar year. vacation requests must be submitted at least 2 weeks in advance through the hr portal.'),
 Document(metadata={'category': 'performance'}, page_content='Questi

In [9]:
# split document into manageable chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
all_chunks = splitter.split_documents(documents)

In [10]:
all_chunks[0]

Document(metadata={'category': 'remote_work'}, page_content="Question: what is the company's policy on remote work?\nAnswer: our company offers a hybrid work model. employees can work remotely up to 3 days per week, with in-office presence required on tuesdays and thursdays for team meetings and collaboration. department heads may adjust this policy based on project needs. employees must ensure they have reliable internet connection and a secure workspace when working remotely.")

In [11]:
len(all_chunks)

46

In [12]:
all_chunks

[Document(metadata={'category': 'remote_work'}, page_content="Question: what is the company's policy on remote work?\nAnswer: our company offers a hybrid work model. employees can work remotely up to 3 days per week, with in-office presence required on tuesdays and thursdays for team meetings and collaboration. department heads may adjust this policy based on project needs. employees must ensure they have reliable internet connection and a secure workspace when working remotely."),
 Document(metadata={'category': 'leave_policy'}, page_content='Question: how many vacation days am i entitled to per year?\nAnswer: full-time employees receive 15 paid vacation days annually, accrued at 1.25 days per month. after 3 years of service, this increases to 20 days. unused vacation days (up to 5) can be carried over to the next calendar year. vacation requests must be submitted at least 2 weeks in advance through the hr portal.'),
 Document(metadata={'category': 'performance'}, page_content='Questi

# Embedding Generation and Storing in Vector Store

In [13]:
# Embeded documents
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")


C:\Users\USER\AppData\Local\Temp\ipykernel_11024\460058110.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(
d:\QA-System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
#Build Faiss vector store
vector_store = FAISS.from_documents(
    documents=all_chunks,
    embedding=embedding_function

)
# vector_store.add_documents(all_chunks)
vector_store.save_local("faiss_index")


In [15]:
vector_store.add_documents(all_chunks)


['52dc2632-6314-45c3-9b65-f781e913f7f6',
 'c5fc4aea-479d-42af-af3b-3aafc4082172',
 '2795186d-6a02-4a6f-a727-56cdd8339aa8',
 '3835c7b1-ceb6-4135-9519-5b0be39db62f',
 'd103c2b3-7918-460a-a705-46c1cda5d978',
 '5eac6499-dd81-40a8-ac67-a81f98e0e582',
 '1405a09f-b2c1-48ea-a1f1-bca2035e0114',
 'e88619b3-adc7-4ddb-bcca-3b52d0276e51',
 '54bd6155-bef5-497f-871a-59d5ea33aab7',
 '9b0cb6e3-eeff-4854-9761-e8a608db0036',
 '7661d80e-0705-495a-bcc3-5d86804ee7f2',
 'c4d65aae-1e37-44f3-a772-8dd6bc08a6ac',
 'c9d15317-34e5-48a2-8964-b0f700bde6e7',
 'b3509cca-d8d6-479c-a5c3-406f341c37d0',
 '92b4a705-1729-4c5f-b520-107d663f3c6c',
 '79ebf8e2-5892-4c37-a070-2489d39fa805',
 '9fa446f2-a9f7-4344-b7f1-be51bbdc0e90',
 'f65cbef8-05e5-4ab2-ac82-11f394afcf31',
 '077a1d14-48e8-4fe1-9ce7-abcf7fe0e939',
 'b5ad3f53-7459-4f73-b280-290223498a9c',
 '04418b05-74b4-4599-a777-884dcbce2e2c',
 'bdbc09d7-62a4-41d7-a28d-9ad152f5c53f',
 '904bd5d4-ed32-48c3-a05c-ad4b1c17f6ff',
 'bad43159-359c-4382-955d-507ef240cd1f',
 'b177b3b1-60ea-

In [16]:
vector_store

# RETRIEVAL

In [17]:
#Build retriever
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 3})

In [18]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001AFA8546E40>, search_kwargs={'k': 3})

In [19]:
retriever.invoke('how does the anual bonus program work')

[Document(id='04171292-42d8-46de-9000-be9db818b6e0', metadata={'category': 'compensation'}, page_content='Question: how does the annual bonus program work?'),
 Document(id='c9d15317-34e5-48a2-8964-b0f700bde6e7', metadata={'category': 'compensation'}, page_content='Question: how does the annual bonus program work?'),
 Document(id='f65b1bdf-0c18-46aa-97e6-d9b517b7ca2e', metadata={'category': 'compensation'}, page_content="Answer: the company's annual bonus program is based on both company and individual performance. bonuses are calculated as a percentage of base salary, ranging from 5-20% depending on position level. the bonus pool is funded based on company financial performance against annual targets. individual bonuses are then determined by personal performance ratings from the year-end review process. employees must be employed on the bonus payment date (typically march 15) to receive payment. new employees")]

# AUGUMENTATION

In [20]:
# prompt design

prompt = PromptTemplate(
    template =  """
        You are a helpful assistant.
        Answer only from the provided context.
        if the context is insufficient, just say I don't know.

        context: {retrieved_text}
        Question: {user_question}
        """,
        input_variables=["retrieved_text", "user_question"]

)

In [21]:
question = "is the parental leave available? if yes then what is benefit"

In [22]:
# retrieved context
retrieved_docs = retriever.invoke(question)
retrieved_docs

[Document(id='a88ad5c9-ba84-4cc5-ac25-4b4d8f4b99bb', metadata={'category': 'leave_policy'}, page_content="Question: what is the company's parental leave policy?"),
 Document(id='ec89156c-683e-483a-821e-1d37554e0c61', metadata={'category': 'leave_policy'}, page_content="Question: what is the company's parental leave policy?"),
 Document(id='6fb27d53-36a2-4d9a-adf4-b2bbfb3e34e5', metadata={'category': 'leave_policy'}, page_content='Answer: the company provides 12 weeks of paid parental leave for primary caregivers and 4 weeks for secondary caregivers following the birth or adoption of a child. this benefit is available to full-time employees who have been with the company for at least 1 year. employees should notify their manager and hr at least 30 days before the anticipated leave date. benefits continue during parental leave, and employees return to their same or equivalent position.')]

In [23]:
retrieved_text = "\n".join(doc.page_content for doc in retrieved_docs)

In [24]:
retrieved_text

"Question: what is the company's parental leave policy?\nQuestion: what is the company's parental leave policy?\nAnswer: the company provides 12 weeks of paid parental leave for primary caregivers and 4 weeks for secondary caregivers following the birth or adoption of a child. this benefit is available to full-time employees who have been with the company for at least 1 year. employees should notify their manager and hr at least 30 days before the anticipated leave date. benefits continue during parental leave, and employees return to their same or equivalent position."

In [25]:
final_prompt = prompt.invoke({"retrieved_text": retrieved_text, "user_question": question}).text

In [26]:
final_prompt

"\n        You are a helpful assistant.\n        Answer only from the provided context.\n        if the context is insufficient, just say I don't know.\n\n        context: Question: what is the company's parental leave policy?\nQuestion: what is the company's parental leave policy?\nAnswer: the company provides 12 weeks of paid parental leave for primary caregivers and 4 weeks for secondary caregivers following the birth or adoption of a child. this benefit is available to full-time employees who have been with the company for at least 1 year. employees should notify their manager and hr at least 30 days before the anticipated leave date. benefits continue during parental leave, and employees return to their same or equivalent position.\n        Question: is the parental leave available? if yes then what is benefit\n        "

# Generation

In [27]:
# Load llm from Huggingfaceendpoint
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-32B",
    task="text-generation",
    temperature=0.8,
    max_new_tokens=250
)

model = ChatHuggingFace(llm=llm)

In [28]:
import re
raw_response = model.invoke([HumanMessage(content=final_prompt)])
raw_response = raw_response.content.strip()

# Response cleaning
def clean_response(raw_response: str) -> str:
    return re.sub(r"<think>.*?</think>", "", raw_response, flags=re.DOTALL).strip()


#Get clean output
cleaned_answer = clean_response(raw_response)
print(cleaned_answer)

Yes, the parental leave is available. The company provides **12 weeks of paid parental leave for primary caregivers** and **4 weeks for secondary caregivers** following the birth or adoption of a child. This benefit applies to full-time employees with at least 1 year of tenure. Employees must notify their manager and HR 30 days in advance, benefits continue during leave, and employees return to their same or equivalent position.


In [29]:
# Fallback logic
def final_response_logic(cleaned: str) -> str:
    if "i don't know" in cleaned.lower() or "cannot answer" in cleaned.lower():
        return "I don't know.\nI couldn’t find information about this. You might try rephrasing your question."

    return cleaned


In [30]:
# Get final output
final_answer = final_response_logic(cleaned_answer)
print(final_answer)

Yes, the parental leave is available. The company provides **12 weeks of paid parental leave for primary caregivers** and **4 weeks for secondary caregivers** following the birth or adoption of a child. This benefit applies to full-time employees with at least 1 year of tenure. Employees must notify their manager and HR 30 days in advance, benefits continue during leave, and employees return to their same or equivalent position.


# Evaluation (Response Accuracy)

In [31]:
# Sample Questions for Evaluation
sample_questions = [
    "Do employees get dental insurance?",
    "What mental health support is available?",
    "Is there a bonus for joining this year?",
    "What’s the bereavement leave duration?",
    "What is the company policy on parental leave?",
    "is the parental leave available?"
]


In [32]:
results = []
#Loop through each question and generate a response

for question in sample_questions:
    #Retrieve relevant documents for the question
    retrieved_docs = retriever.invoke(question)
    retrieved_text = "\n".join(doc.page_content for doc in retrieved_docs)

    # Generate prompt and model response
    final_prompt = prompt.invoke({"retrieved_text": retrieved_text, "user_question": question}).text
    raw_response = model.invoke([HumanMessage(content=final_prompt)])
    
    # Clean and process final response
    raw_response = raw_response.content.strip()
    cleaned = clean_response(raw_response)
    final_answer = final_response_logic(cleaned)

    # Append result for evaluation
    results.append({"question": question, "answer": final_answer})
    print(f"Q: {question}\nA: {final_answer}\n{'-'*80}")


Q: Do employees get dental insurance?
A: I don't know.
I couldn’t find information about this. You might try rephrasing your question.
--------------------------------------------------------------------------------
Q: What mental health support is available?
A: The company offers mental health support through its Employee Assistance Program (EAP), which provides up to 8 free confidential counseling sessions annually per issue. The health insurance plan includes mental health coverage with a $25 copay for in-network providers. Employees also have free access to the Headspace meditation app, and the company supports mental health awareness days and allows mental health days as needed.
--------------------------------------------------------------------------------
Q: Is there a bonus for joining this year?
A: New employees who join before October 1 are eligible for a prorated bonus based on months of service. If joining after October 1, there is no bonus eligibility.
-------------------

In [33]:
#save all responses 
with open("qa_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("Results saved to qa_results.json ")

Results saved to qa_results.json 


Load Reaults and Categorize Performances

In [34]:
import re
from nltk.tokenize import word_tokenize

In [35]:
# Load saved  qa results
with open("qa_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

In [36]:
# Categorize responses based on content of the answer
total = len(results)
has_answer = 0
dont_know = 0

for r in results:
    answer = r["answer"].lower()
    if "i don't know" in answer or "cannot answer" in answer or "couldn't find information" in answer:
        r["response_type"] = "don't know"
        dont_know += 1
    else:
        r["response_type"] = "has answer"
        has_answer += 1

In [37]:
# Calculate metrics
answer_percentage = (has_answer / total) * 100
dont_know_percentage = (dont_know / total) * 100

print(f"\nTotal questions: {total}")
print(f"Questions answered: {has_answer} ({answer_percentage:.2f}%)")
print(f"Questions without answer: {dont_know} ({dont_know_percentage:.2f}%)")


Total questions: 6
Questions answered: 5 (83.33%)
Questions without answer: 1 (16.67%)


In [38]:

# Save updated results
with open("qa_evaluation.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("Evaluation results saved to qa_evaluation.json")

Evaluation results saved to qa_evaluation.json


Building a Chain With RetrievalQA

In [39]:
from langchain.chains import RetrievalQA

In [40]:
# prompt template
'''
I can't use prevous prompt because langchain uses context and question as input and when i use previous one it throws error
'''
prompt1 = PromptTemplate(
    template="""
        You are a helpful assistant.
        Answer only from the provided context.
        If the context is insufficient, just say I don't know.

        context: {context}
        Question: {question}
    """,
    input_variables=["context", "question"]
)

In [41]:
# RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=model,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt1}
)

In [42]:
# Run a query
response = qa_chain.run("What are the benefits of joining this company?")
print(response)

C:\Users\USER\AppData\Local\Temp\ipykernel_11024\2080802981.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa_chain.run("What are the benefits of joining this company?")


<think>
Okay, let me tackle these questions. The user is asking about mental health resources and the benefits of joining the company. First, I need to check the provided context.

Looking at the context, the answer talks about professional development programs, a stipend for courses, internal workshops, cross-departmental projects, and performance reviews. But there's no mention of mental health resources. The first question is specifically about mental health, which isn't covered here. So for that, I should say I don't know.

For the second question about benefits, the context lists several things: the stipend, monthly workshops, cross-departmental projects, and individualized development plans. These are all part of professional growth, so I can list those as benefits. But since the context doesn't mention other benefits like health insurance, vacation days, or wellness programs, I shouldn't assume those. I'll stick strictly to what's provided. The user wants answers only from the c

Building a Chain using Runnable

In [43]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [44]:
def format_docs(retrieved_docs):
    retrieved_text = "\n".join(doc.page_content for doc in retrieved_docs)
    return retrieved_text

In [45]:
'''
parallel chain is used as we have to input one is context and another one is user_question
'''
parallel_chain = RunnableParallel({
    'retrieved_text':retriever | RunnableLambda(format_docs),
    'user_question': RunnablePassthrough()
})


In [46]:
parallel_chain.invoke('Do employees get dental insurance?')

{'retrieved_text': "Answer: the company provides several mental health resources for employees. our employee assistance program (eap) offers confidential counseling services with up to 8 free sessions per issue annually. the health insurance plan includes mental health coverage with in-network providers requiring only a $25 copay. employees also have access to the headspace meditation app at no cost. the company observes mental health awareness days and encourages employees to take mental health days as needed\nAnswer: the company provides several mental health resources for employees. our employee assistance program (eap) offers confidential counseling services with up to 8 free sessions per issue annually. the health insurance plan includes mental health coverage with in-network providers requiring only a $25 copay. employees also have access to the headspace meditation app at no cost. the company observes mental health awareness days and encourages employees to take mental health da

In [47]:
parser = StrOutputParser()

In [48]:
final_chain = parallel_chain | prompt | model | parser

In [49]:
final_chain.invoke('what are the benefit in joining this company?')

'<think>\nOkay, let\'s see. The user is asking about the mental health resources provided by the company and the benefits of joining this company. I need to answer based only on the given context.\n\nFirst, for the mental health resources question. The context provided talks about professional development programs like a stipend for courses, internal workshops, cross-departmental projects, and discussions during performance reviews. But there\'s no mention of mental health resources. The context doesn\'t include things like counseling services, employee assistance programs, mental health days, or any related benefits. So I can\'t infer anything beyond what\'s written. Therefore, the answer should be "I don\'t know" because the context doesn\'t provide that information.\n\nNext, the benefits of joining the company. The context does list several professional development opportunities. These include the $1,500 stipend, monthly workshops, cross-departmental projects, and individualized dev